# 时间序列分类

In [ ]:
import pprint
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from reservoir_computing.modules import RC_model
from reservoir_computing.utils import compute_test_scores
from reservoir_computing.datasets import ClfLoader

np.random.seed(0) # 为了可重复性

## 配置 RC 模型

In [ ]:
config = {}

# 储层的超参数
config['n_internal_units'] = 500        # 储层的大小
config['spectral_radius'] = 0.59        # 储层的最大特征值
config['leak'] = 0.6                    # 储层状态更新中的泄漏量（None 或 1.0 表示无泄漏）
config['connectivity'] = 0.25           # 储层中非零连接的百分比
config['input_scaling'] = 0.1           # 输入权重的缩放
config['noise_level'] = 0.01            # 储层状态更新中的噪声
config['n_drop'] = 5                    # 要丢弃的瞬态状态数
config['bidir'] = True                  # 如果为 True，使用双向储层
config['circle'] = False                # 使用圆形拓扑的储层

# 降维超参数
config['dimred_method'] = 'tenpca'      # 选项：{None（无降维）, 'pca', 'tenpca'}
config['n_dim'] = 75                    # 降维过程后的结果维度数

# MTS 表示类型
config['mts_rep'] = 'reservoir'         # MTS 表示：{'last', 'mean', 'output', 'reservoir'}
config['w_ridge_embedding'] = 10.0      # 岭回归的正则化参数

# 读出层类型
config['readout_type'] = 'lin'          # 用于分类的读出层：{'lin', 'mlp', 'svm'}
config['w_ridge'] = 5.0                 # 岭回归读出层的正则化

pprint.pprint(config)

{'bidir': True,
 'circle': False,
 'connectivity': 0.25,
 'dimred_method': 'tenpca',
 'input_scaling': 0.1,
 'leak': 0.6,
 'mts_rep': 'reservoir',
 'n_dim': 75,
 'n_drop': 5,
 'n_internal_units': 500,
 'noise_level': 0.01,
 'readout_type': 'lin',
 'spectral_radius': 0.59,
 'w_ridge': 5.0,
 'w_ridge_embedding': 10.0}


## 准备数据

In [3]:
Xtr, Ytr, Xte, Yte = ClfLoader().get_data('Japanese_Vowels')

Loaded Japanese_Vowels dataset.
Number of classes: 9
Data shapes:
  Xtr: (270, 29, 12)
  Ytr: (270, 1)
  Xte: (370, 29, 12)
  Yte: (370, 1)


In [ ]:
# 标签的独热编码
onehot_encoder = OneHotEncoder(sparse_output=False)
Ytr = onehot_encoder.fit_transform(Ytr)
Yte = onehot_encoder.transform(Yte)

## 初始化、训练和评估 RC 模型

In [5]:
classifier =  RC_model(**config)

In [ ]:
# 训练模型
tr_time = classifier.fit(Xtr, Ytr) 

Training completed in 0.01 min


In [ ]:
# 计算测试数据的预测
pred_class = classifier.predict(Xte) 
accuracy, f1 = compute_test_scores(pred_class, Yte)
print(f"Accuracy = {accuracy:.3f}, F1 = {f1:.3f}")

Accuracy = 0.981, F1 = 0.981
